# Final pipeline: model <-> remeshing to t = 300

| Component | Owner | What it does here |
| --- | --- | --- |
| Statistics MLP | P3 | Predicts solid fraction, tip position and interface length from (delta, epsilonM, D, t); not run here |
| Surrogate model (GNN) | P1 design, P2 training | Latent design: GNN encoder -> latent stepper -> GNN decoder on any mesh |
| Remeshing and field transfer | P1 | Builds the next adaptive mesh from the predicted phase (refine within distance 8 of the interface) |
| Trust check | P3 | Compares the MLP statistics with those of the model fields; this notebook writes the model side (`trust_check_inputs.json`) |

The pipeline starts from the seed in the undercooled melt and the parameters (delta, epsilonM, D), advances the latent frame by frame, and at the end of every 10-frame epoch decodes on the current mesh and lets remeshing build the next one. Only these 30 decodes are needed, so the measured time is the real online cost.

Reported on the 10 validation cases (test and outer cases are P4's final run): area-weighted solid IoU at t = 100, 200, 300 (the proposal's main metric), phase and temperature RMSE, solid-area error, tip position error, and time per case.

Needs the trained model files in `model_dir` (`autoencoder.pt`, `stepper.pt`, `latent_normalization.pt`). Run through HTCondor, not on the access point.

In [ ]:
cache_dir = 'cache'
split_path = 'splits.json'
model_dir = 'models'
out_dir = 'outputs/pipeline'
solver_seconds = (32.0, 76.0)

In [ ]:
import json
import os
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'P1'))
from amr.remesh import DISTANCE
from amr.data import load_splits
from amr.pipeline import load_model, run_case

os.makedirs(out_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
split = load_splits(split_path)
ae, stepper, mean, std = load_model(model_dir, device)
print(device, 'remesh distance:', DISTANCE)

In [ ]:
rows, timing = [], {}
for case in split['val']:
    r, t = run_case(ae, stepper, mean, std, cache_dir, case, device)
    rows += r
    timing[case] = t
    last = r[-1]
    print(f"{case}: IoU at t=300 {last['solid_iou']:.3f}, phase RMSE {last['phi_rmse_percent']:.2f}%, "
          f"{t['total']:.1f} s", flush=True)

## Proposal metrics on the 10 validation cases

In [ ]:
summary = {}
for f in (100, 200, 300):
    sub = [r for r in rows if r['frame'] == f]
    summary[f't={f}'] = {
        'solid_iou_min': float(min(r['solid_iou'] for r in sub)),
        'solid_iou_median': float(np.median([r['solid_iou'] for r in sub])),
        'phase_rmse_percent_median_worst': [float(np.median([r['phi_rmse_percent'] for r in sub])),
                                            float(max(r['phi_rmse_percent'] for r in sub))],
        'u_rmse_over_delta_percent_median_worst': [float(np.median([r['u_rmse_over_delta_percent'] for r in sub])),
                                                   float(max(r['u_rmse_over_delta_percent'] for r in sub))],
        'solid_area_percent_median_worst': [float(np.median([abs(r['solid_area_percent']) for r in sub])),
                                            float(max(abs(r['solid_area_percent']) for r in sub))],
        'tip_position_error_median_worst': [float(np.nanmedian([r['tip_position_error_max'] for r in sub])),
                                            float(np.nanmax([r['tip_position_error_max'] for r in sub]))]}
totals = [t['total'] for t in timing.values()]
summary['seconds_per_case'] = {'median': float(np.median(totals)), 'max': float(max(totals)),
                               'parts_median': {k: float(np.median([t[k] for t in timing.values()]))
                                                for k in ('latent', 'decode', 'remesh')}}
summary['speed_up_vs_solver'] = [solver_seconds[0] / np.median(totals), solver_seconds[1] / np.median(totals)]
print(json.dumps(summary, indent=1))

## Inputs for the trust check (P3)

`trust_check_inputs.json` holds, for every case and decoded frame, the solid fraction and the mean arm-tip radius of the model field (`stats_pred`) and of the solver field on the same mesh (`stats_true`). The trust check compares `stats_pred` with the MLP prediction for the same (delta, epsilonM, D, t) and raises the flag when they disagree; `stats_true` lets P3 calibrate that threshold on the validation cases.

In [ ]:
with open(os.path.join(out_dir, 'trust_check_inputs.json'), 'w') as fh:
    json.dump([{k: r[k] for k in ('case', 'frame', 'stats_pred', 'stats_true')} for r in rows], fh, indent=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for case in split['val']:
    rr = [r for r in rows if r['case'] == case]
    f = [r['frame'] for r in rr]
    axes[0].plot(f, [r['solid_iou'] for r in rr], lw=1, label=case)
    axes[1].plot(f, [r['tip_position_error_max'] for r in rr], lw=1)
axes[0].set_title('Solid IoU along the model <-> remeshing rollout')
axes[1].set_title('Largest arm-tip position error')
for ax in axes:
    ax.set_xlabel('frame (time)')
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'pipeline.png'), dpi=120)
with open(os.path.join(out_dir, 'report.json'), 'w') as fh:
    json.dump({'summary': summary, 'timing': timing,
               'rows': [{k: v for k, v in r.items() if not k.startswith('stats')} for r in rows]}, fh, indent=1)